### mounting layers | making the medallion architecture

In [0]:
# config for mounting
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope="nobel-laureates-kv", key="client-id"),
    "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope="nobel-laureates-kv", key="secret-key"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{dbutils.secrets.get(scope="nobel-laureates-kv", key="tenant-id")}/oauth2/token"
}

In [0]:
# mounting the bronze, silver and the gold layers
dbutils.fs.mount(
    source = "abfss://bronze@nobellaureatesstorageacc.dfs.core.windows.net/",
    mount_point = "/mnt/bronze_layer",
    extra_configs = configs
)

dbutils.fs.mount(
    source = "abfss://silver@nobellaureatesstorageacc.dfs.core.windows.net/",
    mount_point = "/mnt/silver_layer",
    extra_configs = configs
)

dbutils.fs.mount(
    source = "abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/",
    mount_point = "/mnt/gold_layer",
    extra_configs = configs
)

In [0]:
display(dbutils.fs.mounts())

### data ingestion | null handling | redundancy removal | data validation | cleansed df

In [0]:
from pyspark.sql.functions import col, count, when, coalesce
from pyspark.sql import functions as sf

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/bronze_layer/complete_raw_data/master_data.csv")

In [0]:
print((df.count(), len(df.columns))) # row x cols

In [0]:
df.printSchema()

In [0]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.drop('Overall motivation')

In [0]:
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.withColumn(
    'Fullname',
    sf.concat(
        sf.coalesce(sf.col('Firstname'), sf.lit('')),
        sf.lit(' '),
        sf.coalesce(sf.col('Surname'), sf.lit(''))
    )
)

In [0]:
rename_dict = {
	'Born country': 'born_country',
	'Born country code': 'born_country_code',
	'Born city': 'born_city',
	'Died country': 'died_country',
	'Died country code': 'died_country_code',
	'Died city': 'died_city',
    'Organization name': 'organization_name',
	'Organization city': 'organization_city',
	'Organization country': 'organization_country'
}

for old_name, new_name in rename_dict.items():
    df = df\
        .withColumnRenamed(old_name, new_name)

In [0]:
print('nulls in the respective cols')

died_cols = ['Died', 'died_country', 'died_country_code', 'died_city']

df.select(*[
    (
        sf.count(sf.when((sf.isnan(c) | sf.col(c).isNull()), c)) if t not in ("timestamp", "date")
        else sf.count(sf.when(sf.col(c).isNull(), c))
    ).alias(c)
    for c, t in df.dtypes if c in died_cols
]).show()

In [0]:
alive = 332
countries_of_dead_people = 347
country_code_dead_people = 347
cities_of_dead_people = 353

nulls_in_country_col_to_fill = nulls_in_country_code_to_fill = countries_of_dead_people - alive
cities_col_to_fill = cities_of_dead_people - alive

print(f'nulls_in_country_col_to_fill: {nulls_in_country_col_to_fill}\nnulls_in_country_code_to_fill: {nulls_in_country_code_to_fill}\ncities_col_to_fill: {cities_col_to_fill}')

In [0]:
df = df.withColumn('died_city', when(col('Died').isNotNull(), col('born_city')).otherwise(col('died_city')))
df = df.withColumn('died_country', when(col('Died').isNotNull(), col('born_country')).otherwise(col('died_country')))
df = df.withColumn('died_country', when(col('Died').isNotNull(), col('born_country')).otherwise(col('died_country')))
df = df.withColumn('died_country_code', when(col('Died').isNotNull(), col('born_country_code')).otherwise(col('died_country_code')))
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
df = df.withColumn('organization_name', when(col('organization_name').isNull(), 'Independent').otherwise(col('organization_name')))
df = df.withColumn('organization_city', when(col('organization_city').isNull(), col('born_city')).otherwise(col('organization_city')))
df = df.withColumn('organization_country', when(col('organization_country').isNull(), col('born_country')).otherwise(col('organization_country')))

In [0]:
df = df.filter(col('born_country').isNotNull())
df = df.filter(col('Born').isNotNull())

In [0]:
print((df.count(), len(df.columns))) # row x cols

In [0]:
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts_row = null_counts.collect()[0].asDict()
filtered_nulls = {k: v for k, v in null_counts_row.items() if v > 0}

print(filtered_nulls)

In [0]:
df_laureates = df.select("Id", "Firstname", "Surname", "Born", "Died", "Gender", "Year", "Category", "Motivation", "Fullname").distinct()
df_countries = df.select("born_country", "born_country_code", "died_country", "died_country_code").distinct()
df_cities = df.select( "born_city", "died_city").distinct()
df_organizations = df.select("organization_name", "organization_city", "organization_country").distinct()

In [0]:
df.write.format("delta").mode("overwrite").save("/mnt/silver_layer/master_df")
df_laureates.write.format("delta").mode("overwrite").save("/mnt/silver_layer/laureates")
df_countries.write.format("delta").mode("overwrite").save("/mnt/silver_layer/countries")
df_cities.write.format("delta").mode("overwrite").save("/mnt/silver_layer/cities")
df_organizations.write.format("delta").mode("overwrite").save("/mnt/silver_layer/organizations")

### ❄ schema | aggs | kpis

In [0]:
dbutils.fs.ls('/mnt/silver_layer')

In [0]:
df = spark.read.format("delta").load('/mnt/silver_layer/master_df')

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Define the column renaming mapping
rename_dict = {
    'Id': 'id',
    'Firstname': 'firstname',
    'Surname': 'surname',
    'Born': 'born',
    'Died': 'died',
    'born_country': 'born_country',
    'born_country_code': 'born_country_code',
    'born_city': 'born_city',
    'died_country': 'died_country',
    'died_country_code': 'died_country_code',
    'died_city': 'died_city',
    'Gender': 'gender',
    'Year': 'year',
    'Category': 'category',
    'Motivation': 'motivation',
    'organization_name': 'organization_name',
    'organization_city': 'organization_city',
    'organization_country': 'organization_country',
    'Fullname': 'name'
}

# Apply the renaming to the DataFrame
df = df.select([F.col(col).alias(rename_dict.get(col, col)) for col in df.columns])

# Show the renamed DataFrame schema
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Step 1: Base cleaning (removing first name, surname)
df = df.drop("firstname", "surname")

# Step 2: Country Dimension
country_df = df.select(
    F.col("born_country").alias("country_name"),
    F.col("born_country_code").alias("country_code")
).union(
    df.select(
        F.col("died_country").alias("country_name"),
        F.col("died_country_code").alias("country_code")
    )
).union(
    df.select(
        F.col("organization_country").alias("country_name"),
        F.lit(None).cast("string").alias("country_code")
    )
).dropna(subset=["country_name"]).dropDuplicates(["country_name"])

country_df = country_df.withColumn("country_id", F.monotonically_increasing_id())

# Add country_id to df for joins
df = df.join(country_df.select("country_name", "country_id").withColumnRenamed("country_name", "born_country"), on="born_country", how="left") \
       .withColumnRenamed("country_id", "born_country_id")

df = df.join(country_df.select("country_name", "country_id").withColumnRenamed("country_name", "died_country"), on="died_country", how="left") \
       .withColumnRenamed("country_id", "died_country_id")

df = df.join(country_df.select("country_name", "country_id").withColumnRenamed("country_name", "organization_country"), on="organization_country", how="left") \
       .withColumnRenamed("country_id", "organization_country_id")

# Step 3: Location Dimension (uses country_id instead of country_name)
location_df = df.select(
    F.col("born_city").alias("city_name"),
    F.col("born_country_id").alias("country_id")
).union(
    df.select(
        F.col("died_city").alias("city_name"),
        F.col("died_country_id").alias("country_id")
    )
).union(
    df.select(
        F.col("organization_city").alias("city_name"),
        F.col("organization_country_id").alias("country_id")
    )
).dropna(subset=["city_name", "country_id"]).dropDuplicates(["city_name", "country_id"]) \
 .withColumn("location_id", F.monotonically_increasing_id())

# Step 4: Organization Dimension
organization_df = df.select("organization_name", "organization_city", "organization_country_id").dropna(subset=["organization_name"]).dropDuplicates(["organization_name"])

organization_df = organization_df.join(location_df, 
    (organization_df.organization_city == location_df.city_name) & 
    (organization_df.organization_country_id == location_df.country_id), 
    "left"
)

organization_df = organization_df.select("organization_name", "location_id").dropDuplicates(["organization_name", "location_id"]) \
    .withColumn("organization_id", F.monotonically_increasing_id())

# Step 5: Laureate Dimension
laureate_dim_df = df.select("id", "year", "category").dropDuplicates(["id", "year", "category"]).withColumnRenamed("id", "laureate_id")

# Step 6: Fact Table

# Join organization_id
df = df.join(organization_df, on="organization_name", how="left")

# Join born_location_id
born_loc = df.select("born_city", "born_country_id").dropDuplicates(["born_city", "born_country_id"])
born_loc = born_loc.join(location_df, 
    (born_loc.born_city == location_df.city_name) & 
    (born_loc.born_country_id == location_df.country_id), 
    "left"
).select("born_city", "born_country_id", "location_id") \
 .withColumnRenamed("location_id", "born_location_id")

df = df.join(born_loc, on=["born_city", "born_country_id"], how="left")

# Join died_location_id
died_loc = df.select("died_city", "died_country_id").dropDuplicates(["died_city", "died_country_id"])
died_loc = died_loc.join(location_df, 
    (died_loc.died_city == location_df.city_name) & 
    (died_loc.died_country_id == location_df.country_id), 
    "left"
).select("died_city", "died_country_id", "location_id") \
 .withColumnRenamed("location_id", "died_location_id")

df = df.join(died_loc, on=["died_city", "died_country_id"], how="left")

# Final fact table
fact_df = df.select(
    F.col("id").alias("laureate_id"),
    "organization_id",
    "born_location_id",
    "died_location_id",
    "name",
    "gender",
    "motivation"
)

In [0]:
print('country dimension:')
country_df.printSchema()
print('location dimension:')
location_df.printSchema()
print('organization dimension:')
organization_df.printSchema()
print('laureate dimension:')
laureate_dim_df.printSchema()
print('fact table:')
fact_df.printSchema()

### data validation

In [0]:
# Fact vs Organization
fact_df.join(organization_df, "organization_id", "left_anti").count()

# Fact vs Location (born)
fact_df.join(location_df.withColumnRenamed("location_id", "born_location_id"), "born_location_id", "left_anti").count()

# Fact vs Location (died)
fact_df.join(location_df.withColumnRenamed("location_id", "died_location_id"), "died_location_id", "left_anti").count()

# Fact vs Laureate
fact_df.join(laureate_dim_df, "laureate_id", "left_anti").count()

In [0]:
fact_df.select("organization_id", "born_location_id", "died_location_id", "laureate_id") \
       .filter("organization_id IS NULL OR born_location_id IS NULL OR died_location_id IS NULL OR laureate_id IS NULL") \
       .count()

In [0]:
print(organization_df.groupBy("organization_id").count().filter("count > 1").count())
print(location_df.groupBy("location_id").count().filter("count > 1").count())
print(country_df.groupBy("country_id").count().filter("count > 1").count())
print(laureate_dim_df.groupBy("laureate_id").count().filter("count > 1").count())

In [0]:
fact_df.select("organization_id").distinct().count() == organization_df.select("organization_id").count()  # should match

In [0]:
# saving the facts and dimensions
# dimensions
country_df.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/country_dimension")
location_df.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/location_dimension")
organization_df.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/organization_dimension")
laureate_dim_df.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/laureate_dimension")

# fact table
fact_df.write.format("delta").mode("overwrite").save("/mnt/gold_layer/facts/fact_table")

### BI implementations

In [0]:
# reading the data from the gold layer
country_dimension = spark.read.format("delta").load("/mnt/gold_layer/dimensions/country_dimension")
location_dimension = spark.read.format("delta").load("/mnt/gold_layer/dimensions/location_dimension")
organization_dimension = spark.read.format("delta").load("/mnt/gold_layer/dimensions/organization_dimension")
laureate_dimension = spark.read.format("delta").load("/mnt/gold_layer/dimensions/laureate_dimension")

# Read the fact table
fact_table = spark.read.format("delta").load("/mnt/gold_layer/facts/fact_table")

In [0]:
print(country_dimension.printSchema())
print(location_dimension.printSchema())
print(organization_dimension.printSchema())
print(laureate_dimension.printSchema())
print(fact_table.printSchema())

### aggregations | kpis

In [0]:
from pyspark.sql import functions as F

agg_laureates_by_gender = fact_table.groupBy("gender").agg(
    F.count("laureate_id").alias("total_laureates")
)

agg_laureates_by_gender.display()

In [0]:
agg_laureates_by_category = laureate_dimension.groupBy("category").agg(
    F.count("laureate_id").alias("total_laureates")
)

agg_laureates_by_category.display()

In [0]:
# Nobel Prize Distribution by Year
agg_nobel_distribution_by_year = laureate_dimension.groupBy("year").agg(
    F.count("laureate_id").alias("total_laureates")
).orderBy("year")

display(agg_nobel_distribution_by_year)

In [0]:
# Top Organizations with the Most Laureates
agg_top_organizations = fact_table.join(organization_dimension, "organization_id").groupBy("organization_name").agg(
    F.count("laureate_id").alias("total_laureates")
).orderBy(F.desc("total_laureates"))

display(agg_top_organizations)

In [0]:
# Top Countries with the Most Laureates
agg_top_countries = fact_table.join(location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]) \
    .join(country_dimension, location_dimension["country_id"] == country_dimension["country_id"]) \
    .groupBy("country_name") \
    .agg(F.count("laureate_id").alias("total_laureates")) \
    .orderBy(F.desc("total_laureates"))

display(agg_top_countries)

In [0]:
kpi_laureates_by_decade = fact_table \
    .join(laureate_dimension, "laureate_id", "left") \
    .withColumn("decade", (F.col("year") / 10).cast("int") * 10) \
    .groupBy("decade") \
    .agg(F.count("laureate_id").alias("total_laureates")) \
    .orderBy("decade")

display(kpi_laureates_by_decade)

In [0]:
kpi_gender_distribution_by_category = fact_table \
    .join(laureate_dimension, "laureate_id", "left") \
    .groupBy("category", "gender") \
    .agg(F.count("laureate_id").alias("total")) \
    .orderBy("category", "gender")

display(kpi_gender_distribution_by_category)

In [0]:
# Category Trend by Each Gender Over the Decade
from pyspark.sql.functions import floor

kpi_category_trend_by_gender = laureate_dimension.join(
    fact_table, "laureate_id"
).withColumn(
    "decade", (floor(laureate_dimension["year"] / 10) * 10)
).groupBy("decade", "gender", "category").count()

display(kpi_category_trend_by_gender)

In [0]:
# Gender Ratio for Each Country
kpi_gender_ratio_by_country = fact_table.join(
    location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]
).join(
    country_dimension, location_dimension["country_id"] == country_dimension["country_id"]
).groupBy("country_name", "gender").count()

display(kpi_gender_ratio_by_country)

In [0]:
from pyspark.sql.functions import min, max, countDistinct
from pyspark.sql.functions import col

org_laureate_years = laureate_dimension.join(
    fact_table, "laureate_id"
).groupBy("organization_id").agg(
    min("year").alias("first_year"),
    max("year").alias("last_year"),
    countDistinct("laureate_id").alias("total_laureates")
).withColumn(
    "active_years", col("last_year") - col("first_year") + 1
).withColumn(
    "impact_score", col("total_laureates") / col("active_years")
)

kpi_organization_impact_score = org_laureate_years
display(kpi_organization_impact_score)

In [0]:
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window

# Join tables to bring in country and category
kpi_country_category = fact_table \
    .join(location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]) \
    .join(country_dimension, location_dimension["country_id"] == country_dimension["country_id"]) \
    .join(laureate_dimension, "laureate_id") \
    .groupBy("country_name", "category") \
    .agg(count("laureate_id").alias("laureate_count"))

# Window to rank the top category per country
window_spec = Window.partitionBy("country_name").orderBy(col("laureate_count").desc())

kpi_top_category_by_country = kpi_country_category \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank")

display(kpi_top_category_by_country)

In [0]:
agg_laureates_by_gender.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_laureates_by_gender")
agg_laureates_by_category.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_laureates_by_category")
agg_nobel_distribution_by_year.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_nobel_distribution_by_year")
agg_top_organizations.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_top_organizations")
agg_top_countries.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_top_countries")

In [0]:
kpi_laureates_by_decade.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_laureates_by_decade")
kpi_gender_distribution_by_category.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_gender_distribution_by_category")
kpi_category_trend_by_gender.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_category_trend_by_gender")
kpi_gender_ratio_by_country.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_gender_ratio_by_country")
kpi_organization_impact_score.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_organization_impact_score")
kpi_top_category_by_country.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_top_category_by_country")

### external tables - unity catalog

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
USE CATALOG db_workspace_4359823114074185;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS aggregations;
CREATE SCHEMA IF NOT EXISTS kpis;

In [0]:
%sql
USE SCHEMA aggregations;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS aggregations.agg_laureates_by_category
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/aggregations/agg_laureates_by_category';

CREATE TABLE IF NOT EXISTS aggregations.agg_laureates_by_gender
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/aggregations/agg_laureates_by_gender';

CREATE TABLE IF NOT EXISTS aggregations.agg_nobel_distribution_by_year
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/aggregations/agg_nobel_distribution_by_year';

CREATE TABLE IF NOT EXISTS aggregations.agg_top_countries
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/aggregations/agg_top_countries';

CREATE TABLE IF NOT EXISTS aggregations.agg_top_organizations
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/aggregations/agg_top_organizations';

In [0]:
%sql
USE SCHEMA kpis;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS kpis.kpi_category_trend_by_gender
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_category_trend_by_gender';

CREATE TABLE IF NOT EXISTS kpis.kpi_gender_distribution_by_category
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_gender_distribution_by_category';

CREATE TABLE IF NOT EXISTS kpis.kpi_gender_ratio_by_country
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_gender_ratio_by_country';

CREATE TABLE IF NOT EXISTS kpis.kpi_laureates_by_decade
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_laureates_by_decade';

CREATE TABLE IF NOT EXISTS kpis.kpi_organization_impact_score
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_organization_impact_score';

CREATE TABLE IF NOT EXISTS kpis.kpi_top_category_by_country
USING DELTA
LOCATION 'abfss://gold@nobellaureatesstorageacc.dfs.core.windows.net/kpis/kpi_top_category_by_country';

In [0]:
%sql
CREATE TABLE kpi_category_trend_by_gender AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_category_trend_by_gender;

CREATE TABLE kpi_gender_distribution_by_category AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_gender_distribution_by_category;

CREATE TABLE kpi_gender_ratio_by_country AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_gender_ratio_by_country;

CREATE TABLE kpi_laureates_by_decade AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_laureates_by_decade;

CREATE TABLE kpi_organization_impact_score AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_organization_impact_score;

CREATE TABLE kpi_top_category_by_country AS SELECT * FROM db_workspace_4359823114074185.kpis.kpi_top_category_by_country;

In [0]:
%sql
CREATE TABLE agg_laureates_by_category AS SELECT * FROM db_workspace_4359823114074185.aggregations.agg_laureates_by_category;

CREATE TABLE agg_laureates_by_gender AS SELECT * FROM db_workspace_4359823114074185.aggregations.agg_laureates_by_gender;

CREATE TABLE agg_nobel_distribution_by_year AS SELECT * FROM db_workspace_4359823114074185.aggregations.agg_nobel_distribution_by_year;

CREATE TABLE agg_top_countries AS SELECT * FROM db_workspace_4359823114074185.aggregations.agg_top_countries;

CREATE TABLE agg_top_organizations AS SELECT * FROM db_workspace_4359823114074185.aggregations.agg_top_organizations;

### visualizations

In [0]:
display(dbutils.fs.ls('/mnt/gold_layer/aggregations'))

In [0]:
agg_laureates_by_category = spark.read.format('delta').load('/mnt/gold_layer/aggregations/agg_laureates_by_category')
display(agg_laureates_by_category)

Databricks visualization. Run in Databricks to view.

In [0]:
agg_laureates_by_gender = spark.read.format('delta').load('/mnt/gold_layer/aggregations/agg_laureates_by_gender')
display(agg_laureates_by_gender)

Databricks visualization. Run in Databricks to view.

In [0]:
agg_nobel_distribution_by_year = spark.read.format('delta').load('/mnt/gold_layer/aggregations/agg_nobel_distribution_by_year')
display(agg_nobel_distribution_by_year)

Databricks visualization. Run in Databricks to view.

In [0]:
agg_top_countries = spark.read.format('delta').load('/mnt/gold_layer/aggregations/agg_top_countries')
display(agg_top_countries)

Databricks visualization. Run in Databricks to view.

In [0]:
agg_top_organizations = spark.read.format('delta').load('/mnt/gold_layer/aggregations/agg_top_organizations')
display(agg_top_organizations)

Databricks visualization. Run in Databricks to view.

In [0]:
display(dbutils.fs.ls('/mnt/gold_layer/kpis'))

In [0]:
kpi_category_trend_by_gender = spark.read.format('delta').load('/mnt/gold_layer/kpis/kpi_category_trend_by_gender')
display(kpi_category_trend_by_gender)

Databricks visualization. Run in Databricks to view.

In [0]:
kpi_gender_distribution_by_category = spark.read.format('delta').load('/mnt/gold_layer/kpis/kpi_gender_distribution_by_category')
display(kpi_gender_distribution_by_category)

Databricks visualization. Run in Databricks to view.

In [0]:
kpi_gender_ratio_by_country = spark.read.format('delta').load('/mnt/gold_layer/kpis/kpi_gender_ratio_by_country')
display(kpi_gender_ratio_by_country)

Databricks visualization. Run in Databricks to view.

In [0]:
kpi_laureates_by_decade = spark.read.format('delta').load('/mnt/gold_layer/kpis/kpi_laureates_by_decade')
display(kpi_laureates_by_decade)

Databricks visualization. Run in Databricks to view.

In [0]:
kpi_organization_impact_score = spark.read.format('delta').load('/mnt/gold_layer/kpis/kpi_organization_impact_score')
display(kpi_organization_impact_score)

Databricks visualization. Run in Databricks to view.

In [0]:
kpi_top_category_by_country = spark.sql('''
    WITH cleaned AS (
        SELECT
            CASE
                WHEN country_name LIKE '%(now %)' THEN
                    TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(country_name, '(now ', -1), ')', 1))
                ELSE country_name
            END AS country_cleaned,
            SUM(laureate_count) AS l_count
        FROM kpis.kpi_top_category_by_country
        GROUP BY
            CASE
                WHEN country_name LIKE '%(now %)' THEN
                    TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(country_name, '(now ', -1), ')', 1))
                ELSE country_name
            END
    )
    SELECT country_cleaned AS country_name, l_count
    FROM cleaned
''')

display(kpi_top_category_by_country)

Databricks visualization. Run in Databricks to view.